# 06 Evaluate Models

Purpose: evaluate trained models on validation or test splits, generate prediction JSON, and compute detection metrics.

**Inputs**

- Prepared detection dataset
- One or more experiment directories under `outputs/experiments/`

**Outputs**

- `metrics.json`
- `per_class_metrics.csv`
- `predictions/predictions.json`
- `predictions/false_positives.csv`
- `predictions/false_negatives.csv`


In [ ]:
from pathlib import Path
import os
import sys

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "scripts").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Could not find repo root. Open the notebook from inside the fishDetect repository.")

REPO = find_repo_root()
os.chdir(REPO)

DATASET_ROOT = Path(os.environ.get(
    "FISHDETECT_DATASET_ROOT",
    "/Users/ec/Documents/Data/FishDetectNOAA/_data/merged_viame_v2"
))

PREPARED_ROOT = Path(os.environ.get(
    "FISHDETECT_PREPARED_ROOT",
    "/Users/ec/Documents/Data/FishDetectNOAA/_data/prepared_ml"
))

os.environ["FISHDETECT_DATASET_ROOT"] = str(DATASET_ROOT)
os.environ["FISHDETECT_PREPARED_ROOT"] = str(PREPARED_ROOT)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

print("Repo root:", REPO)
print("Dataset root:", DATASET_ROOT)
print("Prepared root:", PREPARED_ROOT)
print("Dataset exists:", DATASET_ROOT.exists())
print("Prepared parent exists:", PREPARED_ROOT.parent.exists())


## Choose Evaluation Scope


In [ ]:
SPLIT = "test"
EVALUATE_ALL = True
EXPERIMENT = "yolo8n_det"
print("Split:", SPLIT)
print("Evaluate all experiments:", EVALUATE_ALL)


## Run Evaluation


In [ ]:
if EVALUATE_ALL:
    !python scripts/evaluate_models.py --config configs/experiments.yaml --all --split $SPLIT
else:
    !python scripts/evaluate_models.py --config configs/experiments.yaml --experiment $EXPERIMENT --split $SPLIT


## Display Metrics


In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

summary_path = Path("outputs/evaluation_summary.json")
if summary_path.exists():
    display(json.loads(summary_path.read_text()))
else:
    print("No evaluation summary found.")

experiment_dirs = sorted((Path("outputs/experiments")).glob("*/metrics.json"))
for metrics_path in experiment_dirs[:5]:
    exp_dir = metrics_path.parent
    metrics = json.loads(metrics_path.read_text())
    display(Markdown(f"### {exp_dir.name}"))
    keys = ["map50", "map50_95", "precision", "recall", "f1", "inference_speed_ms", "model_size_mb"]
    display(pd.DataFrame([{k: metrics.get(k) for k in keys}]))
    for name in ["per_class_metrics.csv", "predictions/false_positives.csv", "predictions/false_negatives.csv"]:
        path = exp_dir / name
        print(f"{'OK' if path.exists() else 'MISSING':8} {path}")
    per_class = exp_dir / "per_class_metrics.csv"
    if per_class.exists():
        display(pd.read_csv(per_class).head(15))


Use these outputs to review mAP50, mAP50-95, precision, recall, F1, per-class performance, rare-class behavior, false positives, false negatives, and object-size performance when available.
